# Einsum: from vector operations to latent attention

Run in order with NumPy. The examples use small arrays and CPU computation.
Each new contraction is followed by an equivalent indexing, sum or matrix product.

In an expression such as `ij,jk->ik`, letters name axes. Inputs appear before
`->`; output axes appear after it, in the requested order. For each output
index, multiply the corresponding input entries and sum over labels absent
from the output. A shared label refers to the same index in both inputs.

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. Keep or sum an axis

In [2]:
a = np.array([1, 2, 3])
a

array([1, 2, 3])

In [3]:
# i stays in the output: keep the entries.
np.einsum('i->i', a)

array([1, 2, 3])

In [4]:
# No output labels: sum over i and return a scalar.
np.einsum('i->', a)

np.int64(6)

In [5]:
# The same reduction using the familiar operation.
a.sum()

np.int64(6)

## 2. Elementwise product and dot product

In [6]:
b = np.array([4, 5, 6])
b

array([4, 5, 6])

In [7]:
# Both inputs use index i; retaining i gives one product per entry.
np.einsum('i,i->i', a, b)

array([ 4, 10, 18])

In [8]:
a * b

array([ 4, 10, 18])

In [9]:
# Omit i from the output: sum those products.
np.einsum('i,i->', a, b)

np.int64(32)

In [10]:
a @ b

np.int64(32)

## 3. Independent axes form an outer product

In [11]:
# i and j vary independently; keep both in the output.
outer = np.einsum('i,j->ij', a, b)
outer

array([[ 4,  5,  6],
       [ 8, 10, 12],
       [12, 15, 18]])

In [12]:
# Each row is one a[i] multiplied by all of b.
a[:, None] * b[None, :]

array([[ 4,  5,  6],
       [ 8, 10, 12],
       [12, 15, 18]])

## 4. Reorder axes

In [13]:
A = np.array([[1, 2, 3], [4, 5, 6]])
A

array([[1, 2, 3],
       [4, 5, 6]])

In [14]:
# Input axes are (i, j); output axes are (j, i).
np.einsum('ij->ji', A)

array([[1, 4],
       [2, 5],
       [3, 6]])

In [15]:
A.T

array([[1, 4],
       [2, 5],
       [3, 6]])

## 5. Matrix multiplication, one output entry at a time

In [16]:
B = np.array([[1, 2], [3, 4], [5, 6]])
B

array([[1, 2],
       [3, 4],
       [5, 6]])

In [17]:
# Output entry [i=0, k=1] uses row 0 of A and column 1 of B.
# j runs through their three matching entries.
A[0, :] * B[:, 1]

array([ 2,  8, 18])

In [18]:
(A[0, :] * B[:, 1]).sum()

np.int64(28)

In [19]:
# Keep i and k. Sum over j, which appears in the inputs only.
matmul = np.einsum('ij,jk->ik', A, B)
matmul

array([[22, 28],
       [49, 64]])

In [20]:
A @ B

array([[22, 28],
       [49, 64]])

In [21]:
# A literal contraction loop, now that the individual entry is established.
loop_matmul = np.zeros((A.shape[0], B.shape[1]), dtype=A.dtype)
for i in range(A.shape[0]):
    for k in range(B.shape[1]):
        for j in range(A.shape[1]):
            loop_matmul[i, k] += A[i, j] * B[j, k]
np.array_equal(loop_matmul, matmul)

True

## 6. A repeated label within one input selects a diagonal

In [22]:
square = np.arange(1, 10).reshape(3, 3)
square

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [23]:
# The two i axes must have the same size; use entries [i, i].
np.einsum('ii->i', square)

array([1, 5, 9])

In [24]:
# Removing i also sums the diagonal: the trace.
np.einsum('ii->', square)

np.int64(15)

## 7. Keep batch axes independent

In [25]:
batched_A = np.stack([A, A + 10])  # [batch=2, i=2, j=3]
batched_A

array([[[ 1,  2,  3],
        [ 4,  5,  6]],

       [[11, 12, 13],
        [14, 15, 16]]])

In [26]:
batched_B = np.stack([B, B * 2])   # [batch=2, j=3, k=2]
batched_B

array([[[ 1,  2],
        [ 3,  4],
        [ 5,  6]],

       [[ 2,  4],
        [ 6,  8],
        [10, 12]]])

In [27]:
# b remains in the output, so each batch item has its own matrix product.
batched_result = np.einsum('bij,bjk->bik', batched_A, batched_B)
batched_result

array([[[ 22,  28],
        [ 49,  64]],

       [[224, 296],
        [278, 368]]])

In [28]:
np.array_equal(batched_result, batched_A @ batched_B)

True

In [29]:
# B has no b axis: reuse the same B for each batch item.
shared_weight_result = np.einsum('bij,jk->bik', batched_A, B)
shared_weight_result

array([[[ 22,  28],
        [ 49,  64]],

       [[112, 148],
        [139, 184]]])

In [30]:
# ... keeps any leading batch dimensions, using NumPy broadcasting.
np.array_equal(np.einsum('...ij,...jk->...ik', batched_A, B), batched_A @ B)

True

## 8. Read an attention contraction

`b`: batch, `h`: head, `q`: query position, `k`: key position,
`d`: query/key features, `v`: value features.

In [31]:
rng = np.random.default_rng(42)
queries = rng.normal(size=(1, 2, 3, 2))
queries.shape  # [b, h, q, d]

(1, 2, 3, 2)

In [32]:
keys = rng.normal(size=(1, 2, 4, 2))
keys.shape     # [b, h, k, d]

(1, 2, 4, 2)

In [33]:
# Compare the first query and all keys in head 0.
queries[0, 0, 0] * keys[0, 0]

array([[ 0.02 , -1.172],
       [ 0.142,  0.894],
       [ 0.112,  0.997],
       [ 0.268,  0.052]])

In [34]:
# Sum d; keep the batch, head, query and key axes.
scores = np.einsum('bhqd,bhkd->bhqk', queries, keys)
scores[0, 0]

array([[-1.152,  1.036,  1.11 ,  0.32 ],
       [ 1.11 , -0.457, -0.625,  0.612],
       [-1.597,  0.207,  0.529, -1.649]])

In [35]:
np.allclose(scores, queries @ keys.swapaxes(-1, -2))

True

In [36]:
# A row of attention weights is a softmax over key positions.
# This example isolates the contraction; all four keys are visible.
scaled_scores = scores / np.sqrt(queries.shape[-1])
exp_scores = np.exp(scaled_scores - scaled_scores.max(axis=-1, keepdims=True))
weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
weights[0, 0]

array([[0.074, 0.349, 0.367, 0.21 ],
       [0.43 , 0.142, 0.126, 0.302],
       [0.1  , 0.357, 0.448, 0.096]])

In [37]:
values = rng.normal(size=(1, 2, 4, 3))
values.shape   # [b, h, k, v]

(1, 2, 4, 3)

In [38]:
# Sum over keys k; keep the value features v for each query.
attended = np.einsum('bhqk,bhkv->bhqv', weights, values)
attended[0, 0]

array([[-0.061,  0.095, -0.03 ],
       [-0.057,  0.005,  0.987],
       [ 0.091,  0.287, -0.065]])

In [39]:
np.allclose(attended, weights @ values)

True

## 9. The first MLA expression: `bhtd,hdr->bhtr`

Here `t` names the query-position axis called `q` above. Labels are local to
an expression. `r` names the shared KV latent features.

In [40]:
q_content = queries  # [batch, head, query position, content feature]
key_up_weights = rng.normal(size=(2, 2, 3))
key_up_weights.shape  # [head, content feature, latent feature]

(2, 2, 3)

In [41]:
# Inspect one head's content-to-latent matrix.
key_up_weights[0]

array([[-1.457, -0.32 , -0.47 ],
       [-0.639, -0.275,  1.495]])

In [42]:
# One head and one query: [d] @ [d, r] gives [r].
q_content[0, 0, 0] @ key_up_weights[0]

array([ 0.22 ,  0.189, -1.698])

In [43]:
# h selects a head's own weights; d is summed; b,h,t,r remain.
latent_queries = np.einsum('bhtd,hdr->bhtr', q_content, key_up_weights)
latent_queries[0, 0, 0]

array([ 0.22 ,  0.189, -1.698])

In [44]:
# Equivalent per-head matrix products stacked along the head axis.
per_head = np.stack([q_content[:, h] @ key_up_weights[h]
                     for h in range(q_content.shape[1])], axis=1)
np.allclose(latent_queries, per_head)

True

## 10. Attend to cached latent vectors

In [45]:
kv_latent = rng.normal(size=(1, 4, 3))
kv_latent.shape  # [batch, key position, latent feature], shared across heads

(1, 4, 3)

In [46]:
# Sum latent features r; keep both positions t and k.
latent_scores = np.einsum('bhtr,bkr->bhtk', latent_queries, kv_latent)
latent_scores[0, 0]

array([[ 0.899,  0.385,  1.066, -1.036],
       [-1.968,  0.154,  1.758, -0.186],
       [ 3.752, -0.66 , -4.855,  1.256]])

In [47]:
# Expand keys, then score them, to verify the same operation.
expanded_keys = np.einsum('bkr,hdr->bhkd', kv_latent, key_up_weights)
expanded_scores = np.einsum('bhtd,bhkd->bhtk', q_content, expanded_keys)
np.allclose(latent_scores, expanded_scores)

True

In [48]:
# Use the attention weights from section 8 to isolate the value algebra.
# Sum past positions k into one latent mixture per query and head.
latent_mixture = np.einsum('bhtk,bkr->bhtr', weights, kv_latent)
latent_mixture[0, 0]

array([[-0.472, -0.028, -0.285],
       [ 0.23 ,  0.363, -0.084],
       [-0.618, -0.109, -0.448]])

## 11. The second MLA expression: `bhtr,hvr->bhtv`

In [49]:
value_up_weights = rng.normal(size=(2, 2, 3))
value_up_weights.shape  # [head, value feature, latent feature]

(2, 2, 3)

In [50]:
# One mixed latent vector [r] projects to value features [v].
latent_mixture[0, 0, 0] @ value_up_weights[0].T

array([0.019, 0.322])

In [51]:
# Sum r; h selects each head's projection; b,h,t,v remain.
mla_heads = np.einsum('bhtr,hvr->bhtv', latent_mixture, value_up_weights)
mla_heads[0, 0, 0]

array([0.019, 0.322])

In [52]:
# Expand all cached values, then weight them: same final heads.
expanded_values = np.einsum('bkr,hvr->bhkv', kv_latent, value_up_weights)
expanded_heads = np.einsum('bhtk,bhkv->bhtv', weights, expanded_values)
np.allclose(mla_heads, expanded_heads)

True

## 12. Diagnose a mislabeled contraction

Read each input's axis labels left to right, then check shared sizes and the
requested output axes. Using different labels for the same batch axis creates
independent axes; dropping a batch label from the output sums it.

In [53]:
# Renaming B's batch label c creates every (b,c) pair of batch items.
all_batch_pairs = np.einsum('bij,cjk->bcik', batched_A, batched_B)
all_batch_pairs.shape

(2, 2, 2, 2)

In [54]:
# Omitting b sums the correctly paired batch products.
summed_batches = np.einsum('bij,bjk->ik', batched_A, batched_B)
np.array_equal(summed_batches, batched_result.sum(axis=0))

True

In [55]:
# A shared label must have compatible dimensions. This j has size 4, not 3.
try:
    np.einsum('ij,jk->ik', A, np.ones((4, 2)))
except ValueError as error:
    shape_error = str(error)
shape_error

'operands could not be broadcast together with remapped shapes [original->remapped]: (2,3)->(2,newaxis,3) (4,2)->(2,4) '

## 13. Combine established contractions

For expressions with several operands, an implementation chooses an order of
pairwise contractions. Intermediate array sizes depend on that order.
The NumPy option below asks it to choose a contraction path. Inspect runtime
on the target backend before comparing equivalent forms.

In [56]:
# Collapse the already-established latent mixture and value projection.
combined_heads = np.einsum('bhtk,bkr,hvr->bhtv', weights, kv_latent,
                           value_up_weights, optimize='greedy')
np.allclose(combined_heads, mla_heads)

True

## Read the same expressions in MLX

Our [MLA implementation](../src/llm_training/models/mla_mlx.py) uses
`mx.einsum` with the same two-operand label expressions. Notebook
[13](13_mla_tutorial.ipynb) connects them to projections, RoPE and cached decoding.
Backend APIs can differ in optional arguments; `optimize` above is a NumPy option.

References: [NumPy einsum](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)
and [MLX einsum](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.einsum.html).